# LeetCode 1804 — Implement Trie II (Prefix Tree) bằng Python

Link đề: https://leetcode.com/problems/implement-trie-ii-prefix-tree/ · Độ khó: **Medium** · Chủ đề: Hash Table, String, Design, Trie

> 🔒 **Bài Premium.** Đề trong notebook được viết lại theo hiểu biết về bài (nguồn: kho lời giải doocs/leetcode). Nếu bạn có tài khoản Premium, nên đối chiếu lại câu chữ và ràng buộc với đề gốc.

Skeleton LeetCode (bài **thiết kế lớp**):

```python
class Trie:
    def __init__(self):
        ...
    def insert(self, word: str) -> None:
        ...
    def countWordsEqualTo(self, word: str) -> int:
        ...
    def countWordsStartingWith(self, prefix: str) -> int:
        ...
    def erase(self, word: str) -> None:
        ...
```

Lộ trình: **list các từ** (quét toàn bộ mỗi truy vấn) → **`Counter` từ** → **bảng băm mọi tiền tố** → **Trie với hai bộ đếm `pass`/`end`** (lời giải nộp) → **Trie có cắt tỉa nút** khi xóa. Mọi cách được kiểm bằng một **bộ chạy chuỗi lệnh** mô phỏng LeetCode trên lệnh ngẫu nhiên seed cố định.

## 1. Mục tiêu học

1. Hiểu cấu trúc **Trie (cây tiền tố)**: mỗi cạnh là một ký tự, mỗi nút là một tiền tố.
2. Gắn **hai bộ đếm** vào mỗi nút: `pass` (số từ đi qua nút = số từ có tiền tố đó) và `end` (số từ kết thúc đúng tại nút) để hỗ trợ **từ lặp lại** và **xóa**.
3. So sánh chi phí của các thiết kế (list, `Counter`, băm tiền tố, Trie) theo **thời gian mỗi thao tác** và **bộ nhớ**.
4. Viết **bộ chạy chuỗi lệnh** kiểu LeetCode (`["Trie","insert",...]`, `[[],["apple"],...]`) và kiểm thử ngẫu nhiên so với cài đặt brute force.
5. Tránh các bẫy Python: **thuộc tính lớp dùng chung** (`children = {}` ở mức class), quên giảm `pass` khi xóa, cắt tỉa nhầm nút.

## 2. Đề bài & ví dụ

Cài đặt lớp `Trie`:
- `Trie()` — khởi tạo cây rỗng.
- `insert(word)` — chèn chuỗi `word` (có thể chèn **nhiều lần** cùng một từ).
- `countWordsEqualTo(word)` — trả về **số lần** `word` đang có trong trie.
- `countWordsStartingWith(prefix)` — trả về **số chuỗi** trong trie có `prefix` là tiền tố (tính cả bản lặp).
- `erase(word)` — xóa **một** bản của `word` (đề bảo đảm `word` đang tồn tại).

**Ví dụ 1:**

```
Input
["Trie","insert","insert","countWordsEqualTo","countWordsStartingWith","erase","countWordsEqualTo","countWordsStartingWith","erase","countWordsStartingWith"]
[[],["apple"],["apple"],["apple"],["app"],["apple"],["apple"],["app"],["apple"],["app"]]
Output
[null,null,null,2,2,null,1,1,null,0]
```

**Ràng buộc:**
- `1 ≤ word.length, prefix.length ≤ 2000`
- chỉ gồm chữ cái tiếng Anh thường
- tổng cộng tối đa `3·10⁴` lời gọi `insert`, `countWordsEqualTo`, `countWordsStartingWith`, `erase`
- mỗi lời gọi `erase(word)` đều bảo đảm `word` đang có trong trie.

In [1]:
import random
import time
from collections import Counter, defaultdict
from typing import List

example_ops = ["Trie", "insert", "insert", "countWordsEqualTo", "countWordsStartingWith", "erase",
               "countWordsEqualTo", "countWordsStartingWith", "erase", "countWordsStartingWith"]
example_args = [[], ["apple"], ["apple"], ["apple"], ["app"], ["apple"], ["apple"], ["app"], ["apple"], ["app"]]
example_out = [None, None, None, 2, 2, None, 1, 1, None, 0]

def run_commands(cls, ops, args):
    """Mô phỏng cách LeetCode chạy bài thiết kế: lệnh đầu là constructor."""
    out = []
    obj = None
    for op, a in zip(ops, args):
        if op == cls.__name__ or op == "Trie":
            obj = cls(*a)
            out.append(None)
        else:
            out.append(getattr(obj, op)(*a))
    return out

print("Kỳ vọng:", example_out)

Kỳ vọng: [None, None, None, 2, 2, None, 1, 1, None, 0]


## 3. Kiến thức nền

### 3.1 Lớp, `__init__` và thuộc tính **đối tượng**

Thuộc tính gán trong `__init__` (qua `self.x = ...`) là **riêng** của mỗi đối tượng. Thuộc tính khai báo ngay trong thân lớp là **dùng chung** cho mọi đối tượng — nếu là `dict`/`list` (khả biến) thì mọi nút sẽ chia sẻ cùng một dict! (Xem Lỗi 5.)

In [2]:
class Good:
    def __init__(self):
        self.children = {}

class Bad:
    children = {}          # thuộc tính LỚP: dùng chung

g1, g2 = Good(), Good()
b1, b2 = Bad(), Bad()
g1.children["a"] = 1
b1.children["a"] = 1
print("Good:", g2.children, "| Bad:", b2.children, "| cùng một dict?", b1.children is b2.children)

Good: {} | Bad: {'a': 1} | cùng một dict? True


### 3.2 `dict` làm bảng con của nút

`node.children.get(c)` trả `None` nếu chưa có cạnh `c` (không ném `KeyError`). `setdefault(c, Node())` tạo nút mới nếu chưa có — nhưng **luôn** tạo đối tượng `Node()` trước khi kiểm (hơi tốn), nên trong vòng nóng ta viết `if c not in d: d[c] = Node()`.

In [3]:
d = {}
print(d.get("x"))
if "x" not in d:
    d["x"] = []
d["x"].append(1)
print(d)

None
{'x': [1]}


### 3.3 `__slots__` để nút nhẹ hơn

Mỗi đối tượng Python thường có một `__dict__` riêng (tốn bộ nhớ). Khai báo `__slots__ = ("children", "pass_cnt", "end_cnt")` bỏ `__dict__`, giúp nút nhỏ hơn và truy cập nhanh hơn — hữu ích khi trie có hàng triệu nút (tổng độ dài từ tới `3·10⁴ · 2000 = 6·10⁷`!).

In [4]:
import sys as _sys
class NodeDict:
    def __init__(self):
        self.children = {}; self.p = 0; self.e = 0
class NodeSlots:
    __slots__ = ("children", "p", "e")
    def __init__(self):
        self.children = {}; self.p = 0; self.e = 0
a, b = NodeDict(), NodeSlots()
print("có __dict__:", hasattr(a, "__dict__"), "| slots có __dict__:", hasattr(b, "__dict__"))
print("kích thước đối tượng (byte, chưa tính dict con):", _sys.getsizeof(a) + _sys.getsizeof(a.__dict__), "vs", _sys.getsizeof(b))

có __dict__: True | slots có __dict__: False
kích thước đối tượng (byte, chưa tính dict con): 344 vs 56


### 3.4 `str.startswith` và `Counter`

`w.startswith(p)` kiểm tiền tố trong `O(len(p))`. `Counter` là dict đếm: `c[w] += 1`, `c[w] -= 1`, truy cập khóa thiếu trả 0.

In [5]:
c = Counter()
c["apple"] += 2
c["apple"] -= 1
print(c["apple"], c["banana"], "apple".startswith("app"))

1 0 True


### 3.5 `getattr` để gọi phương thức theo tên

Bộ chạy lệnh dùng `getattr(obj, "insert")("apple")` — gọi phương thức theo chuỗi tên, đúng như cách LeetCode đọc input.

In [6]:
class Demo:
    def hello(self, name):
        return "xin chào " + name
print(getattr(Demo(), "hello")("Trie"))

xin chào Trie


## 4. Ý tưởng nền tảng / trực giác

Chèn `"apple"`, `"apple"`, `"app"`, `"ant"` vào trie. Mỗi nút ghi `p` (số từ đi qua — tức có tiền tố này) và `e` (số từ kết thúc ở đây):

```
(root)
  └─ a  p=4
      ├─ p  p=3
      │   └─ p  p=3  e=1        <- "app" kết thúc ở đây
      │       └─ l  p=2
      │           └─ e  p=2  e=2   <- "apple" x2
      └─ n  p=1
          └─ t  p=1  e=1        <- "ant"
```

- `countWordsStartingWith("app")` = `p` của nút `a→p→p` = **3** (`app`, `apple`, `apple`).
- `countWordsEqualTo("apple")` = `e` của nút cuối = **2**.
- `erase("apple")`: đi lại đường `a,p,p,l,e`, giảm `p` mỗi nút trên đường đi, giảm `e` ở nút cuối.

Mấu chốt: **mỗi thao tác chỉ đi dọc một đường dài `L`** ⇒ `O(L)`, không phụ thuộc số từ trong trie.

## 5. Cách 1 — Brute Force / baseline: list các từ

**Ý tưởng:** lưu mọi từ (kể cả lặp) trong một list. Đếm bằng `list.count`, tiền tố bằng quét `startswith`, xóa bằng `list.remove` (xóa một bản). Đúng hiển nhiên ⇒ dùng làm **chuẩn đối chiếu** trong test ngẫu nhiên.

In [7]:
class TrieBrute:
    def __init__(self):
        self.words = []

    def insert(self, word: str) -> None:
        self.words.append(word)

    def countWordsEqualTo(self, word: str) -> int:
        return self.words.count(word)

    def countWordsStartingWith(self, prefix: str) -> int:
        return sum(1 for w in self.words if w.startswith(prefix))

    def erase(self, word: str) -> None:
        self.words.remove(word)

print(run_commands(TrieBrute, example_ops, example_args) == example_out)

True


**Độ phức tạp:** gọi `N` = số từ đang có, `L` = độ dài từ/tiền tố. `insert` `O(1)`; ba thao tác còn lại `O(N·L)`. Với `3·10⁴` lời gọi và `L = 2000`, worst-case cỡ `3·10⁴ · 1.5·10⁴ · 2000` ≈ `10¹²` — quá chậm.

**Nút thắt:** mỗi truy vấn **quét lại toàn bộ** tập từ.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1:** `countWordsEqualTo` chỉ cần **đếm theo từ** ⇒ bảng băm `word → count`, `O(L)` (băm chuỗi).

**Insight 2:** `countWordsStartingWith(p)` = số từ có tiền tố `p`. Nếu mỗi lần chèn từ `w` ta **cộng 1 cho mọi tiền tố** của `w`, thì truy vấn là tra một bảng. Trie làm đúng việc đó nhưng **chia sẻ** các tiền tố chung thành các nút, nên chèn chỉ tốn `O(L)` thay vì `O(L²)` (băm từng tiền tố).

**Quy tắc cho Trie:**
- `insert(w)`: đi từ gốc theo từng ký tự, *nếu* thiếu nút *thì* tạo; mỗi nút đi tới: `pass += 1`; nút cuối: `end += 1`.
- `erase(w)`: đi lại cùng đường, `pass -= 1` mỗi nút, `end -= 1` ở nút cuối (đề bảo đảm từ tồn tại).
- truy vấn: đi theo `w`/`p`; *nếu* gặp cạnh thiếu *thì* trả 0; ngược lại trả `end`/`pass` của nút cuối.

**Invariant:** với mọi nút `u` ứng với chuỗi `s`: `u.pass` = số từ (tính lặp) trong trie có tiền tố `s`; `u.end` = số bản của đúng `s`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — `Counter` theo từ

In [8]:
class TrieCounter:
    def __init__(self):
        self.cnt = Counter()

    def insert(self, word: str) -> None:
        self.cnt[word] += 1

    def countWordsEqualTo(self, word: str) -> int:
        return self.cnt[word]

    def countWordsStartingWith(self, prefix: str) -> int:
        return sum(c for w, c in self.cnt.items() if w.startswith(prefix))

    def erase(self, word: str) -> None:
        self.cnt[word] -= 1
        if self.cnt[word] == 0:
            del self.cnt[word]

print(run_commands(TrieCounter, example_ops, example_args) == example_out)

True


**Độ phức tạp:** `insert`, `countWordsEqualTo`, `erase` `O(L)`; `countWordsStartingWith` `O(D·L)` với `D` = số từ **khác nhau**. Tốt hơn khi nhiều từ trùng, nhưng truy vấn tiền tố vẫn quét.

### 7.2 Cách 3 — Bảng băm mọi tiền tố

In [9]:
class TriePrefixHash:
    def __init__(self):
        self.word_cnt = Counter()
        self.pref_cnt = Counter()

    def insert(self, word: str) -> None:
        self.word_cnt[word] += 1
        for i in range(1, len(word) + 1):
            self.pref_cnt[word[:i]] += 1          # O(i) để cắt + băm

    def countWordsEqualTo(self, word: str) -> int:
        return self.word_cnt[word]

    def countWordsStartingWith(self, prefix: str) -> int:
        return self.pref_cnt[prefix]

    def erase(self, word: str) -> None:
        self.word_cnt[word] -= 1
        for i in range(1, len(word) + 1):
            self.pref_cnt[word[:i]] -= 1

print(run_commands(TriePrefixHash, example_ops, example_args) == example_out)

True


**Độ phức tạp:** truy vấn `O(L)`, nhưng `insert`/`erase` `O(L²)` (cắt và băm `L` tiền tố, tổng độ dài `L(L+1)/2`) và bộ nhớ `O(Σ L²)`. Với `L = 2000`, một lần chèn đã tạo ~2 triệu ký tự chuỗi con. Nút thắt: **không chia sẻ** phần tiền tố chung ⇒ Trie.

### 7.3 Cách 4 — Trie với `pass`/`end` (lời giải chuẩn)

In [10]:
class TrieNode:
    __slots__ = ("children", "pass_cnt", "end_cnt")
    def __init__(self):
        self.children = {}
        self.pass_cnt = 0
        self.end_cnt = 0

class TrieDict:
    def __init__(self):
        self.root = TrieNode()

    def insert(self, word: str) -> None:
        node = self.root
        for c in word:
            if c not in node.children:
                node.children[c] = TrieNode()
            node = node.children[c]
            node.pass_cnt += 1
        node.end_cnt += 1

    def _find(self, s: str):
        node = self.root
        for c in s:
            node = node.children.get(c)
            if node is None:
                return None
        return node

    def countWordsEqualTo(self, word: str) -> int:
        node = self._find(word)
        return node.end_cnt if node else 0

    def countWordsStartingWith(self, prefix: str) -> int:
        node = self._find(prefix)
        return node.pass_cnt if node else 0

    def erase(self, word: str) -> None:
        node = self.root
        for c in word:
            node = node.children[c]
            node.pass_cnt -= 1
        node.end_cnt -= 1

print(run_commands(TrieDict, example_ops, example_args) == example_out)

True


**Độ phức tạp:** mọi thao tác `O(L)`. Bộ nhớ `O(tổng số ký tự đã chèn)` nút (tối đa `6·10⁷` trên lý thuyết, thực tế ít hơn nhờ chia sẻ tiền tố).

Một nút có `pass_cnt == 0` sau khi xóa thì "chết" (không từ nào đi qua) nhưng vẫn chiếm bộ nhớ. Truy vấn qua nút chết vẫn đúng (trả 0) vì mọi nút con của nó cũng có `pass = 0`.

### 7.4 Cách 5 — Trie có cắt tỉa nút khi xóa

Khi `erase` làm `pass` của một nút về 0, ta **xóa cạnh** dẫn tới nút đó khỏi cha — cả nhánh con bên dưới cũng bị bỏ (chúng đều có `pass = 0`). Giữ trie gọn khi có nhiều chèn/xóa xen kẽ.

In [11]:
class TriePrune(TrieDict):
    def erase(self, word: str) -> None:
        node = self.root
        for c in word:
            child = node.children[c]
            child.pass_cnt -= 1
            if child.pass_cnt == 0:          # không còn từ nào đi qua: cắt cả nhánh
                del node.children[c]
                return                       # end_cnt của nút cuối thuộc nhánh đã cắt
            node = child
        node.end_cnt -= 1

    def node_count(self):
        stack, k = [self.root], 0
        while stack:
            u = stack.pop(); k += 1
            stack.extend(u.children.values())
        return k

print(run_commands(TriePrune, example_ops, example_args) == example_out)
t = TriePrune()
for w in ["apple", "app", "apply"]:
    t.insert(w)
print("số nút:", t.node_count())
t.erase("apply"); print("sau erase('apply'):", t.node_count())
t.erase("apple"); print("sau erase('apple'):", t.node_count(), "| countWordsEqualTo('app') =", t.countWordsEqualTo("app"))

True
số nút: 7
sau erase('apply'): 6
sau erase('apple'): 4 | countWordsEqualTo('app') = 1


Tại sao `return` sớm được? Nếu nút `child` có `pass = 0` thì mọi nút sâu hơn trên đường đi cũng có `pass` (trước khi giảm) bằng 1 — chúng chỉ chứa đúng từ này — nên bỏ cả nhánh là đúng, không cần giảm tiếp.

**Độ phức tạp:** vẫn `O(L)` mỗi thao tác; bộ nhớ `O(tổng độ dài các từ đang tồn tại)`.

## 8. TRACE từng bước

Trace Cách 4 trên Ví dụ 1. Chỉ ghi các nút trên đường `a→p→p→l→e` dạng `p/e`:

| lệnh | `a` | `ap` | `app` | `appl` | `apple` | trả về |
|---|---|---|---|---|---|---|
| insert apple | 1/0 | 1/0 | 1/0 | 1/0 | 1/1 | null |
| insert apple | 2/0 | 2/0 | 2/0 | 2/0 | 2/2 | null |
| countWordsEqualTo apple | | | | | **e=2** | 2 |
| countWordsStartingWith app | | | **p=2** | | | 2 |
| erase apple | 1/0 | 1/0 | 1/0 | 1/0 | 1/1 | null |
| countWordsEqualTo apple | | | | | e=1 | 1 |
| countWordsStartingWith app | | | p=1 | | | 1 |
| erase apple | 0/0 | 0/0 | 0/0 | 0/0 | 0/0 | null |
| countWordsStartingWith app | | | p=0 | | | 0 |

In [12]:
class TrieTrace(TrieDict):
    def dump_path(self, s):
        node, parts, pref = self.root, [], ""
        for c in s:
            node = node.children.get(c)
            pref += c
            if node is None:
                parts.append(f"{pref}:—"); break
            parts.append(f"{pref}:{node.pass_cnt}/{node.end_cnt}")
        return " ".join(parts)

def trace_commands(ops, args, path="apple"):
    obj = None
    for op, a in zip(ops, args):
        if op == "Trie":
            obj = TrieTrace(); print("Trie()"); continue
        res = getattr(obj, op)(*a)
        print(f"{op}({a[0]!r}) -> {res} | đường {path}: {obj.dump_path(path)}")

trace_commands(example_ops, example_args)

Trie()
insert('apple') -> None | đường apple: a:1/0 ap:1/0 app:1/0 appl:1/0 apple:1/1
insert('apple') -> None | đường apple: a:2/0 ap:2/0 app:2/0 appl:2/0 apple:2/2
countWordsEqualTo('apple') -> 2 | đường apple: a:2/0 ap:2/0 app:2/0 appl:2/0 apple:2/2
countWordsStartingWith('app') -> 2 | đường apple: a:2/0 ap:2/0 app:2/0 appl:2/0 apple:2/2
erase('apple') -> None | đường apple: a:1/0 ap:1/0 app:1/0 appl:1/0 apple:1/1
countWordsEqualTo('apple') -> 1 | đường apple: a:1/0 ap:1/0 app:1/0 appl:1/0 apple:1/1
countWordsStartingWith('app') -> 1 | đường apple: a:1/0 ap:1/0 app:1/0 appl:1/0 apple:1/1
erase('apple') -> None | đường apple: a:0/0 ap:0/0 app:0/0 appl:0/0 apple:0/0
countWordsStartingWith('app') -> 0 | đường apple: a:0/0 ap:0/0 app:0/0 appl:0/0 apple:0/0


## 9. Code nộp LeetCode

In [13]:
class Trie:
    class _Node:
        __slots__ = ("children", "pass_cnt", "end_cnt")
        def __init__(self):
            self.children = {}
            self.pass_cnt = 0      # số từ có tiền tố = chuỗi của nút này
            self.end_cnt = 0       # số từ kết thúc đúng tại nút này

    def __init__(self):
        self.root = Trie._Node()

    def insert(self, word: str) -> None:
        node = self.root
        for c in word:
            nxt = node.children.get(c)
            if nxt is None:
                nxt = node.children[c] = Trie._Node()
            node = nxt
            node.pass_cnt += 1
        node.end_cnt += 1

    def _find(self, s: str):
        node = self.root
        for c in s:
            node = node.children.get(c)
            if node is None:
                return None
        return node

    def countWordsEqualTo(self, word: str) -> int:
        node = self._find(word)
        return node.end_cnt if node is not None else 0

    def countWordsStartingWith(self, prefix: str) -> int:
        node = self._find(prefix)
        return node.pass_cnt if node is not None else 0

    def erase(self, word: str) -> None:
        node = self.root
        for c in word:
            node = node.children[c]
            node.pass_cnt -= 1
        node.end_cnt -= 1

assert run_commands(Trie, example_ops, example_args) == example_out
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

Gọi `M` là đa tập các từ hiện có. Với mỗi nút `u` ứng với chuỗi `s(u)` (đường đi từ gốc), ta chứng minh bất biến:

- `pass(u) = |{w ∈ M : s(u) là tiền tố của w}|` (tính lặp),
- `end(u) = |{w ∈ M : w = s(u)}|`.

*Khởi tạo:* trie rỗng, mọi bộ đếm 0. ✔

*`insert(w)`:* các nút có `s(u)` là tiền tố khác rỗng của `w` là **đúng** các nút trên đường đi theo `w` (mỗi tiền tố một nút). Ta tăng `pass` của đúng các nút đó và tăng `end` của nút `s(u) = w`. ✔

*`erase(w)`:* `w ∈ M` nên đường đi tồn tại; giảm đúng các bộ đếm đã tăng khi chèn một bản `w`. ✔ (Cách 5: cắt nhánh có `pass = 0` không làm sai bất biến vì các nút bị cắt ứng với tiền tố không còn từ nào; truy vấn vào chúng trả 0 như mong muốn.)

*Truy vấn:* nếu đường đi theo `p` đứt thì không nút nào ứng với `p` ⇒ không từ nào có tiền tố `p` (vì mọi tiền tố của từ đã chèn đều có nút) ⇒ trả 0 đúng. Ngược lại đọc `pass`/`end` theo bất biến. ✔

## 11. Phân tích Time / Space

Gọi `L` = độ dài chuỗi của lời gọi, `N` = số từ hiện có, `D` = số từ khác nhau, `S` = tổng độ dài các từ đã chèn.

| Cách | insert | countEqual | countStartingWith | erase | Space |
|---|---|---|---|---|---|
| 1. List | `O(1)` | `O(N·L)` | `O(N·L)` | `O(N·L)` | `O(S)` |
| 2. Counter | `O(L)` | `O(L)` | `O(D·L)` | `O(L)` | `O(S)` |
| 3. Băm tiền tố | `O(L²)` | `O(L)` | `O(L)` | `O(L²)` | `O(Σ L²)` |
| 4. Trie | `O(L)` | `O(L)` | `O(L)` | `O(L)` | `O(S)` nút |
| 5. Trie cắt tỉa | `O(L)` | `O(L)` | `O(L)` | `O(L)` | `O(độ dài các từ còn sống)` |

Đo một chuỗi lệnh lớn (`3·10⁴` lời gọi, từ dài tới 100, bảng chữ nhỏ để có nhiều tiền tố chung):

In [14]:
def gen_ops(n_ops, rng, alpha="abc", max_len=6, pool_size=30):
    pool = ["".join(rng.choice(alpha) for _ in range(rng.randint(1, max_len))) for _ in range(pool_size)]
    ops, args = ["Trie"], [[]]
    alive = Counter()
    for _ in range(n_ops):
        r = rng.random()
        if r < 0.35 or not alive:
            w = rng.choice(pool); ops.append("insert"); args.append([w]); alive[w] += 1
        elif r < 0.55:
            w = rng.choice([x for x, c in alive.items() if c > 0]); ops.append("erase"); args.append([w])
            alive[w] -= 1
            if alive[w] == 0: del alive[w]
        elif r < 0.75:
            ops.append("countWordsEqualTo"); args.append([rng.choice(pool)])
        else:
            w = rng.choice(pool); ops.append("countWordsStartingWith"); args.append([w[:rng.randint(1, len(w))]])
    return ops, args

rng = random.Random(1804)
big_ops, big_args = gen_ops(30000, rng, alpha="ab", max_len=100, pool_size=2000)
results = {}
for cls in [TrieCounter, TriePrefixHash, TrieDict, TriePrune, Trie]:
    t = time.perf_counter()
    results[cls.__name__] = run_commands(cls, big_ops, big_args)
    print(f"{cls.__name__:15s}: {1000*(time.perf_counter()-t):7.0f} ms")
assert len({tuple(v) for v in results.values()}) == 1
print("mọi cài đặt cho cùng kết quả")

TrieCounter    :     303 ms
TriePrefixHash :     179 ms


TrieDict       :     120 ms
TriePrune      :     138 ms


Trie           :     120 ms
mọi cài đặt cho cùng kết quả


## 12. Bộ test

### 12.1 Ví dụ đề + edge case (chuỗi lệnh tự dựng, đối chiếu brute force)

In [15]:
impls = [TrieCounter, TriePrefixHash, TrieDict, TriePrune, Trie]
edge_cases = [
    (example_ops, example_args),
    # tiền tố dài hơn mọi từ, tiền tố không tồn tại, từ là tiền tố của từ khác
    (["Trie", "insert", "insert", "countWordsStartingWith", "countWordsStartingWith", "countWordsEqualTo",
      "countWordsEqualTo", "erase", "countWordsStartingWith", "countWordsEqualTo"],
     [[], ["app"], ["apple"], ["applesauce"], ["b"], ["ap"], ["app"], ["app"], ["app"], ["apple"]]),
    # một ký tự, chèn/xóa nhiều lần
    (["Trie", "insert", "insert", "insert", "erase", "countWordsEqualTo", "countWordsStartingWith", "erase",
      "erase", "countWordsStartingWith", "insert", "countWordsStartingWith"],
     [[], ["a"], ["a"], ["a"], ["a"], ["a"], ["a"], ["a"], ["a"], ["a"], ["a"], ["a"]]),
    # từ rất dài
    (["Trie", "insert", "countWordsStartingWith", "countWordsEqualTo", "erase", "countWordsStartingWith"],
     [[], ["z" * 2000], ["z" * 1999], ["z" * 2000], ["z" * 2000], ["z"]]),
]
for ops, args in edge_cases:
    want = run_commands(TrieBrute, ops, args)
    for cls in impls:
        assert run_commands(cls, ops, args) == want, cls.__name__
    print("OK:", want)

OK: [None, None, None, 2, 2, None, 1, 1, None, 0]
OK: [None, None, None, 0, 0, 0, 1, None, 1, 1]
OK: [None, None, None, None, None, 2, 2, None, None, 0, None, 1]
OK: [None, None, 1, 1, None, 0]


### 12.2 Chuỗi lệnh ngẫu nhiên seed cố định, so với brute force

In [16]:
rng = random.Random(2021)
total_ops = 0
for trial in range(300):
    ops, args = gen_ops(rng.randint(1, 60), rng, alpha=rng.choice(["ab", "abc", "a"]), max_len=5, pool_size=12)
    want = run_commands(TrieBrute, ops, args)
    for cls in impls:
        assert run_commands(cls, ops, args) == want, (cls.__name__, ops, args)
    total_ops += len(ops)
print("đã đối chiếu", total_ops, "lệnh trên 300 chuỗi ngẫu nhiên")

đã đối chiếu 9252 lệnh trên 300 chuỗi ngẫu nhiên


### 12.3 Kiểm tra bất biến `pass`/`end` trực tiếp

In [17]:
def check_invariant(trie, multiset):
    stack = [(trie.root, "")]
    while stack:
        u, s = stack.pop()
        if s:
            assert u.pass_cnt == sum(c for w, c in multiset.items() if w.startswith(s)), s
            assert u.end_cnt == multiset[s], s
        for ch, v in u.children.items():
            stack.append((v, s + ch))

rng = random.Random(5)
for _ in range(100):
    t, M = Trie(), Counter()
    for _ in range(40):
        if +M and rng.random() < 0.4:
            w = rng.choice(list(+M)); t.erase(w); M[w] -= 1
        else:
            w = "".join(rng.choice("ab") for _ in range(rng.randint(1, 4))); t.insert(w); M[w] += 1
        check_invariant(t, M)
print("bất biến pass/end đúng sau mọi thao tác")

bất biến pass/end đúng sau mọi thao tác


## 13. Lỗi người mới hay gặp

### Lỗi 1 — `erase` chỉ giảm `end`, quên giảm `pass`

In [18]:
class WrongEraseEndOnly(TrieDict):
    def erase(self, word):
        node = self._find(word)
        node.end_cnt -= 1                 # SAI: pass của các nút trên đường không đổi

ops = ["Trie", "insert", "erase", "countWordsStartingWith"]
args = [[], ["apple"], ["apple"], ["app"]]
print("sai:", run_commands(WrongEraseEndOnly, ops, args), "| đúng:", run_commands(Trie, ops, args))

sai: [None, None, None, 1] | đúng: [None, None, None, 0]


**Vì sao sai:** `pass` là số từ đi qua nút; xóa một từ phải giảm `pass` trên **toàn bộ** đường đi. **Sửa:** đi lại từ gốc và `pass -= 1` từng nút.

### Lỗi 2 — Tăng `pass` ở nút **cha** (trước khi bước xuống)

In [19]:
class WrongPassOnParent(TrieDict):
    def insert(self, word):
        node = self.root
        for c in word:
            node.pass_cnt += 1            # SAI: tăng ở nút hiện tại rồi mới xuống con
            if c not in node.children:
                node.children[c] = TrieNode()
            node = node.children[c]
        node.end_cnt += 1

ops = ["Trie", "insert", "countWordsStartingWith"]
args = [[], ["app"], ["app"]]
print("sai:", run_commands(WrongPassOnParent, ops, args), "| đúng:", run_commands(Trie, ops, args))

sai: [None, None, 0] | đúng: [None, None, 1]


**Vì sao sai:** bộ đếm bị lệch một tầng: nút `app` không được tăng (từ kết thúc tại đó), nên `countWordsStartingWith("app")` trả 0. Từ **bằng đúng** tiền tố cũng phải được đếm. **Sửa:** bước xuống con **rồi** mới tăng.

### Lỗi 3 — Cắt tỉa theo `end == 0` thay vì `pass == 0`

In [20]:
class WrongPrune(TrieDict):
    def erase(self, word):
        path = []
        node = self.root
        for c in word:
            path.append((node, c))
            node = node.children[c]
            node.pass_cnt -= 1
        node.end_cnt -= 1
        parent, c = path[-1]
        if node.end_cnt == 0:             # SAI: còn từ khác đi qua nút này
            del parent.children[c]

ops = ["Trie", "insert", "insert", "erase", "countWordsEqualTo", "countWordsStartingWith"]
args = [[], ["app"], ["apple"], ["app"], ["apple"], ["app"]]
print("sai:", run_commands(WrongPrune, ops, args), "| đúng:", run_commands(Trie, ops, args))

sai: [None, None, None, None, 0, 0] | đúng: [None, None, None, None, 1, 1]


**Vì sao sai:** xóa `"app"` làm `end` của nút `app` về 0 nhưng `"apple"` vẫn đi qua nút đó (`pass = 1`). Cắt nút làm mất `"apple"`. **Sửa:** chỉ cắt khi `pass == 0` (Cách 5).

### Lỗi 4 — Truy vấn tiền tố không tồn tại bằng `children[c]`

In [21]:
class WrongFind(TrieDict):
    def countWordsStartingWith(self, prefix):
        node = self.root
        for c in prefix:
            node = node.children[c]       # SAI: KeyError khi cạnh không tồn tại
        return node.pass_cnt

ops = ["Trie", "insert", "countWordsStartingWith"]
args = [[], ["apple"], ["banana"]]
try:
    print(run_commands(WrongFind, ops, args))
except KeyError as e:
    print("KeyError:", e, "| đúng:", run_commands(Trie, ops, args))

KeyError: 'b' | đúng: [None, None, 0]


**Vì sao sai:** `countWordsStartingWith` / `countWordsEqualTo` có thể hỏi chuỗi chưa từng chèn. **Sửa:** `children.get(c)` và trả 0 khi `None`. (Riêng `erase` dùng `children[c]` được vì đề bảo đảm từ tồn tại.)

### Lỗi 5 — Khai báo `children = {}` ở mức lớp

In [22]:
class SharedNode:
    children = {}                         # SAI: một dict dùng chung cho MỌI nút
    def __init__(self):
        self.pass_cnt = 0; self.end_cnt = 0

class WrongShared:
    def __init__(self):
        self.root = SharedNode()
    def insert(self, word):
        node = self.root
        for c in word:
            if c not in node.children:
                node.children[c] = SharedNode()
            node = node.children[c]
            node.pass_cnt += 1
        node.end_cnt += 1
    def countWordsStartingWith(self, prefix):
        node = self.root
        for c in prefix:
            node = node.children.get(c)
            if node is None: return 0
        return node.pass_cnt

t = WrongShared()
t.insert("ab")
print("sai: countWordsStartingWith('b') =", t.countWordsStartingWith("b"), "| đúng: 0")
SharedNode.children.clear()               # dọn dẹp dict dùng chung để không ảnh hưởng cell khác

sai: countWordsStartingWith('b') = 1 | đúng: 0


**Vì sao sai:** mọi nút dùng chung một dict, nên cạnh `'b'` (đáng lẽ là con của nút `a`) cũng "là con" của gốc. Trie sụp thành một bảng phẳng lộn xộn. **Sửa:** gán `self.children = {}` trong `__init__`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Đếm/tra theo tiền tố"**, autocomplete, từ điển ⇒ Trie.
- **Cần đếm lặp và xóa** ⇒ mỗi nút giữ `pass` (đi qua) và `end` (kết thúc). Đây là nâng cấp chuẩn từ LeetCode 208 (Trie I chỉ có cờ `is_end`).
- Truy vấn: `get` + trả 0 khi đứt; cập nhật: bước xuống con rồi mới tăng/giảm.
- Python: dùng `__slots__`, `dict` con (bảng chữ lớn) hoặc list 26 phần tử (bảng chữ nhỏ cố định).
- Bài thiết kế: luôn viết **bộ chạy lệnh** + **cài đặt brute force** để test ngẫu nhiên.

## 15. Biến thể của bài

1. **LeetCode 208 — Implement Trie:** chỉ `insert`, `search`, `startsWith` (cờ `is_end`).
2. **LeetCode 211 — Add and Search Word:** tìm kiếm có ký tự đại diện `.` ⇒ DFS trên trie.
3. **LeetCode 677 — Map Sum Pairs:** mỗi từ có giá trị, hỏi tổng giá trị theo tiền tố — thay `pass` bằng tổng giá trị.
4. **Đếm theo hậu tố:** trie trên từ đảo ngược (Bài tập 5); cả tiền tố **và** hậu tố: LeetCode 745 *Prefix and Suffix Search*.
5. **Trie nhị phân (bit trie):** LeetCode 421 *Maximum XOR of Two Numbers* — cùng kỹ thuật `pass` để hỗ trợ xóa số.

## 16. So sánh các cách

| Cách | Ý tưởng | Ưu điểm | Nhược điểm | Khi nào dùng |
|---|---|---|---|---|
| 1. List | lưu thô | đơn giản, chuẩn đối chiếu | mọi truy vấn `O(N·L)` | test |
| 2. Counter | đếm theo từ | đếm bằng nhau `O(L)` | tiền tố `O(D·L)` | ít truy vấn tiền tố |
| 3. Băm tiền tố | đếm mọi tiền tố | truy vấn `O(L)` | chèn `O(L²)`, tốn bộ nhớ | từ ngắn |
| 4. Trie `pass/end` | chia sẻ tiền tố | mọi thao tác `O(L)` | nhiều đối tượng nút | **nộp bài** |
| 5. Trie cắt tỉa | như 4 + xóa nhánh chết | bộ nhớ theo từ còn sống | code phức tạp hơn | chèn/xóa xen kẽ nhiều |

## 17. 5 bài tập từ dễ đến khó

**Bài 1 (đoán output):** Chuỗi lệnh `insert("ab"), insert("abc"), insert("ab"), countWordsStartingWith("ab"), countWordsEqualTo("ab"), erase("ab"), countWordsStartingWith("abc")` trả về những gì (bỏ `null`)?

**Bài 2 (tính tay):** Sau khi chèn `["a", "ab", "ab", "b"]`, ghi `pass/end` của các nút `a`, `ab`, `b`.

**Bài 3 (điền TODO):** Thêm phương thức `wordsWithPrefix(prefix)` trả về **list** mọi từ (kể cả lặp) có tiền tố `prefix`, theo thứ tự từ điển.

**Bài 4 (tự viết lại):** Cài đặt `TrieArray` lưu cây bằng **các mảng phẳng** (`children[node][c]` là list 26 số nguyên, `-1` nếu thiếu; `pass_[node]`, `end_[node]`) thay vì đối tượng nút.

**Bài 5 (mini challenge):** Cài đặt `TrieSuffix` có thêm `countWordsEndingWith(suffix)` (đếm từ có hậu tố `suffix`) — vẫn `O(L)` mỗi thao tác.

In [23]:
class TrieWithList(Trie):
    def wordsWithPrefix(self, prefix):
        # TODO: tìm nút của prefix, DFS theo thứ tự ký tự, mỗi nút thêm end_cnt bản
        pass

class TrieArray:
    # TODO: các mảng phẳng
    pass

class TrieSuffix:
    # TODO: hai trie — một trên từ, một trên từ đảo ngược
    pass

## 18. Gợi ý 3 mức cho bài tập

| Bài | Gợi ý 1 | Gợi ý 2 | Gợi ý 3 |
|---|---|---|---|
| 1 | `"ab"` là tiền tố của chính nó | Trước khi xóa: 3 từ có tiền tố `ab` | Kết quả `[3, 2, 1]` |
| 2 | Nút `a`: `"a"`, `"ab"`, `"ab"` đi qua | `a: 3/1`, `ab: 2/2` | `b: 1/1` |
| 3 | `_find(prefix)` | DFS đệ quy hoặc stack, duyệt `sorted(children)` | Thêm `[s] * end_cnt` trước khi xuống con |
| 4 | Nút 0 là gốc | Tạo nút: append một list `[-1]*26` và hai số 0 | `idx = ord(c) - 97` |
| 5 | Hậu tố của `w` = tiền tố của `w[::-1]` | Chèn/xóa vào cả hai trie | `countWordsEndingWith(s)` = `pass` của `s[::-1]` trong trie đảo |

## 19. Đáp án tham khảo

In [24]:
ops1 = ["Trie", "insert", "insert", "insert", "countWordsStartingWith", "countWordsEqualTo", "erase", "countWordsStartingWith"]
args1 = [[], ["ab"], ["abc"], ["ab"], ["ab"], ["ab"], ["ab"], ["abc"]]
r1 = [x for x in run_commands(Trie, ops1, args1) if x is not None]
print("Bài 1:", r1); assert r1 == [3, 2, 1]
t2 = TrieTrace()
for w in ["a", "ab", "ab", "b"]:
    t2.insert(w)
print("Bài 2:", t2.dump_path("ab"), "|", t2.dump_path("b"))
assert t2.dump_path("ab") == "a:3/1 ab:2/2" and t2.dump_path("b") == "b:1/1"

Bài 1: [3, 2, 1]
Bài 2: a:3/1 ab:2/2 | b:1/1


In [25]:
class TrieWithList(Trie):
    def wordsWithPrefix(self, prefix):
        node = self._find(prefix)
        out = []
        if node is None:
            return out
        stack = [(node, prefix)]
        while stack:
            u, s = stack.pop()
            out.extend([s] * u.end_cnt)
            for ch in sorted(u.children, reverse=True):   # đảo để pop theo thứ tự tăng
                stack.append((u.children[ch], s + ch))
        return out

class TrieArray:
    def __init__(self):
        self.ch = [[-1] * 26]
        self.p = [0]
        self.e = [0]

    def _new(self):
        self.ch.append([-1] * 26); self.p.append(0); self.e.append(0)
        return len(self.p) - 1

    def insert(self, word):
        u = 0
        for c in word:
            k = ord(c) - 97
            if self.ch[u][k] == -1:
                self.ch[u][k] = self._new()
            u = self.ch[u][k]
            self.p[u] += 1
        self.e[u] += 1

    def _find(self, s):
        u = 0
        for c in s:
            u = self.ch[u][ord(c) - 97]
            if u == -1:
                return -1
        return u

    def countWordsEqualTo(self, word):
        u = self._find(word)
        return 0 if u == -1 else self.e[u]

    def countWordsStartingWith(self, prefix):
        u = self._find(prefix)
        return 0 if u == -1 else self.p[u]

    def erase(self, word):
        u = 0
        for c in word:
            u = self.ch[u][ord(c) - 97]
            self.p[u] -= 1
        self.e[u] -= 1

rng = random.Random(8)
for _ in range(200):
    ops, args = gen_ops(rng.randint(1, 50), rng, alpha="abc", max_len=4, pool_size=10)
    assert run_commands(TrieArray, ops, args) == run_commands(TrieBrute, ops, args)
    t, B = TrieWithList(), TrieBrute()
    for op, a in zip(ops[1:], args[1:]):
        getattr(t, op)(*a); getattr(B, op)(*a)
    for p in ["a", "ab", "b", "c", "abc"]:
        assert t.wordsWithPrefix(p) == sorted(w for w in B.words if w.startswith(p))
demo = TrieWithList()
for w in ["apple", "app", "apple", "ant", "bee"]:
    demo.insert(w)
print("Bài 3, 4: OK | wordsWithPrefix('a') =", demo.wordsWithPrefix("a"))

Bài 3, 4: OK | wordsWithPrefix('a') = ['ant', 'app', 'apple', 'apple']


In [26]:
class TrieSuffix:
    def __init__(self):
        self.fwd = Trie()
        self.rev = Trie()

    def insert(self, word):
        self.fwd.insert(word); self.rev.insert(word[::-1])

    def erase(self, word):
        self.fwd.erase(word); self.rev.erase(word[::-1])

    def countWordsEqualTo(self, word):
        return self.fwd.countWordsEqualTo(word)

    def countWordsStartingWith(self, prefix):
        return self.fwd.countWordsStartingWith(prefix)

    def countWordsEndingWith(self, suffix):
        return self.rev.countWordsStartingWith(suffix[::-1])

rng = random.Random(13)
for _ in range(200):
    T, B = TrieSuffix(), TrieBrute()
    for _ in range(30):
        if B.words and rng.random() < 0.3:
            w = rng.choice(B.words); T.erase(w); B.erase(w)
        else:
            w = "".join(rng.choice("ab") for _ in range(rng.randint(1, 4))); T.insert(w); B.insert(w)
        s = "".join(rng.choice("ab") for _ in range(rng.randint(1, 3)))
        assert T.countWordsEndingWith(s) == sum(1 for x in B.words if x.endswith(s))
print("Bài 5 (biến thể tự dựng — đối chiếu lại đề gốc nếu có): OK")

Bài 5 (biến thể tự dựng — đối chiếu lại đề gốc nếu có): OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
class Node:
    __slots__ = ("children", "pass_cnt", "end_cnt")
    def __init__(self): self.children = {}; self.pass_cnt = 0; self.end_cnt = 0

def insert(w):                       # xuống con RỒI tăng
    u = root
    for c in w:
        if c not in u.children: u.children[c] = Node()
        u = u.children[c]
        u.pass_cnt += 1
    u.end_cnt += 1

def find(s):                         # get + None khi đứt
    u = root
    for c in s:
        u = u.children.get(c)
        if u is None: return None
    return u
# equal = end_cnt, startsWith = pass_cnt; erase: giảm pass trên đường, giảm end ở cuối
```

**Checklist:**
- [ ] `self.children = {}` trong `__init__` (không ở mức lớp)?
- [ ] Bước xuống con rồi mới `pass += 1` (từ bằng tiền tố vẫn được đếm)?
- [ ] `erase` giảm `pass` trên **mọi** nút của đường đi và `end` ở nút cuối?
- [ ] Truy vấn chuỗi không tồn tại trả 0 (không `KeyError`)?
- [ ] Nếu cắt tỉa: chỉ cắt khi `pass == 0`.